# E2 — Transformer Encoder Baseline for RUL Prediction on C-MAPSS FD001

**Study:** Agentic Multi-Machine Predictive Maintenance (AMMPM) using Time-Series Foundation Models for Explainable and Trustworthy RUL Prediction.

**Purpose:** Isolate the effect of self-attention versus recurrence for Remaining Useful Life (RUL) prediction. This notebook is a controlled variant of `E1_lstm_baseline.ipynb`: identical data pipeline (FD001, `SEQUENCE_LENGTH=30` windows, seed 42, engine-level 80/20 split), identical training regimen (50 epochs, Adam lr=1e-3, MSE loss, batch size 256), and an identical prediction head (a linear layer on the final time step's representation) — the **only** thing that changes is the sequence encoder itself: a 2-layer Transformer encoder in place of the 2-layer LSTM. Any RMSE/MAE/PHM difference between E1 and E2 is therefore attributable to the encoder architecture, not to a confound in data, training, or evaluation.

**Reproducibility contract:** global seed fixed to 42 via `src/utils/seed.py::set_global_seed` (called before any data loading or model construction), CPU-only execution for bit-exact determinism, and a dedicated seeded generator for minibatch shuffling. Restart the kernel and *Run All* to reproduce every number in this notebook exactly.


## 0. Setup

Import shared project utilities from `src/` and `configs/`, then fix the global seed as the very first executable step.


In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset


def _find_project_root(start: Path) -> Path:
    """Walk upward from `start` until a directory containing configs/paths.py is found."""
    for candidate in (start, *start.parents):
        if (candidate / "configs" / "paths.py").exists():
            return candidate
    raise RuntimeError(
        "Could not locate the AMMPM project root (no configs/paths.py found above "
        f"{start}). Launch Jupyter from the project root or notebooks/ directory."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from configs.paths import RESULTS_DIR, ensure_dirs  # noqa: E402
from src.data.cmapss_loader import get_cmapss  # noqa: E402
from src.utils.seed import set_global_seed  # noqa: E402

SEED = 42
set_global_seed(SEED)
ensure_dirs()

print(f"Project root: {PROJECT_ROOT}")
print(f"Global seed:  {SEED}")


Project root: /home/bruce-wayne-2005/industrial-ai-project
Global seed:  42


## 1. Data: NASA C-MAPSS FD001

Same subset, same preprocessing as E1: 100 turbofan engines run to failure under a single operating condition/fault mode, min-max normalized (fit on train only), RUL capped at 125 cycles. See `E1_lstm_baseline.ipynb` §1 for the full rationale.


In [2]:
DATA = get_cmapss(fd_num=1, max_rul=125)
train_df = DATA["train_df"]
test_df = DATA["test_df"]
feature_columns = DATA["feature_columns"]

print(f"FD001 train: {train_df['unit_number'].nunique()} engines, {len(train_df)} rows")
print(f"FD001 test:  {test_df['unit_number'].nunique()} engines, {len(test_df)} rows")
print(f"Feature channels ({len(feature_columns)}): {feature_columns}")


FD001 train: 100 engines, 20631 rows
FD001 test:  100 engines, 13096 rows
Feature channels (24): ['op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21']


## 2. Windowing (identical to E1)

`SEQUENCE_LENGTH = 30` cycles per window, chosen because it is at or below the shortest trajectory in both splits (128 train / 31 test), so no engine needs synthetic padding. Every valid window per engine for training; only the final window per engine for test, matching the PHM08 scoring protocol.


In [3]:
SEQUENCE_LENGTH = 30  # cycles per input window


def build_windows(df: pd.DataFrame, feature_cols, window: int, mode: str):
    """Slice per-engine trajectories into fixed-length sliding windows.

    mode="train": every valid window per engine (overlapping sub-trajectory
        augmentation, standard for C-MAPSS sequence-model baselines).
    mode="last": only the final window per engine (one prediction per test
        engine, matching the official PHM08 scoring protocol).
    """
    X_list, y_list = [], []
    for _, group in df.groupby("unit_number"):
        group = group.sort_values("time_in_cycles")
        feats = group[feature_cols].to_numpy(dtype=np.float32)
        rul = group["RUL"].to_numpy(dtype=np.float32)
        n = len(group)

        if n < window:
            # Left-pad short trajectories by repeating the earliest reading,
            # so every engine yields at least one full-length window.
            pad = np.repeat(feats[:1], window - n, axis=0)
            feats = np.concatenate([pad, feats], axis=0)
            rul = np.concatenate([np.repeat(rul[:1], window - n), rul])
            n = window

        if mode == "last":
            X_list.append(feats[-window:])
            y_list.append(rul[-1])
        else:
            for end in range(window, n + 1):
                X_list.append(feats[end - window:end])
                y_list.append(rul[end - 1])

    return np.stack(X_list).astype(np.float32), np.array(y_list, dtype=np.float32)


## 3. Train/validation split — by engine, not by row (identical to E1)

Splitting *rows* would leak overlapping windows of the same engine trajectory across the train/val boundary. We split the 100 training **engine units** 80/20 (same `random_state=SEED` as E1, so E1 and E2 train/validate on exactly the same engines) and window within each split.


In [4]:
all_units = sorted(train_df["unit_number"].unique())
train_units, val_units = train_test_split(
    all_units, test_size=0.2, random_state=SEED, shuffle=True
)

train_split_df = train_df[train_df["unit_number"].isin(train_units)]
val_split_df = train_df[train_df["unit_number"].isin(val_units)]

X_train, y_train = build_windows(train_split_df, feature_columns, SEQUENCE_LENGTH, mode="train")
X_val, y_val = build_windows(val_split_df, feature_columns, SEQUENCE_LENGTH, mode="train")
X_test, y_test = build_windows(test_df, feature_columns, SEQUENCE_LENGTH, mode="last")

print(f"Train engines: {len(train_units)}  -> {X_train.shape[0]} windows")
print(f"Val engines:   {len(val_units)}  -> {X_val.shape[0]} windows")
print(f"Test engines:  {X_test.shape[0]}  -> {X_test.shape[0]} windows (one per engine)")
print(f"Window shape:  (sequence_length={X_train.shape[1]}, n_features={X_train.shape[2]})")


Train engines: 80  -> 14241 windows
Val engines:   20  -> 3490 windows
Test engines:  100  -> 100 windows (one per engine)
Window shape:  (sequence_length=30, n_features=24)


In [5]:
device = torch.device("cpu")  # forced for bit-exact reproducibility (see src/utils/seed.py)


def to_tensor_dataset(X, y):
    return TensorDataset(torch.from_numpy(X), torch.from_numpy(y).unsqueeze(1))


train_dataset = to_tensor_dataset(X_train, y_train)
val_dataset = to_tensor_dataset(X_val, y_val)
test_dataset = to_tensor_dataset(X_test, y_test)

BATCH_SIZE = 256
# Dedicated, explicitly-seeded generator for minibatch shuffling: decouples
# batch order from whatever else has drawn on the global torch RNG earlier
# in the notebook, so results stay identical even if cells above are edited.
shuffle_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    generator=shuffle_generator, num_workers=0,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Device: {device}")
print(f"Batches per epoch (train): {len(train_loader)}")


Device: cpu
Batches per epoch (train): 56


## 4. Model: Transformer encoder baseline

Sensor windows are not natively token sequences the way text is, so three adaptations carry the encoder architecture over to RUL regression:

1. **Input projection** — a linear layer lifts each cycle's 24 raw feature channels into the model's `d_model=64` embedding space (self-attention needs a fixed embedding width, unlike the LSTM which consumes raw features directly).
2. **Positional encoding** — self-attention is permutation-invariant, so without an explicit position signal the encoder cannot tell a window's first cycle from its last. We add the standard fixed sinusoidal encoding (Vaswani et al., 2017) to the projected embeddings before the encoder layers.
3. **Prediction head** — to keep this a controlled comparison against E1, we reuse the *same* pooling strategy as the LSTM baseline: a linear layer reads off the encoded representation at the window's final cycle (rather than, say, mean-pooling over the sequence). The encoder layers are not causally masked, so that final position's representation has already attended over the full window — it is a legitimate summary token, and it isolates the encoder swap as the only architectural change relative to E1.

Configuration: `d_model=64`, `nhead=4`, `num_encoder_layers=2`, `dim_feedforward=128`, `dropout=0.1`.


In [6]:
class PositionalEncoding(nn.Module):
    """Fixed sinusoidal positional encoding (Vaswani et al., 2017)."""

    def __init__(self, d_model: int, max_len: int = 500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, : x.size(1), :]


class TransformerRULRegressor(nn.Module):
    """Transformer-encoder regressor mapping a sensor-window to a scalar RUL."""

    def __init__(
        self,
        input_size: int,
        d_model: int = 64,
        nhead: int = 4,
        num_encoder_layers: int = 2,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        max_len: int = SEQUENCE_LENGTH,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len=max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
        self.head = nn.Linear(d_model, 1)

    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoding(x)
        out = self.encoder(x)
        last_step = out[:, -1, :]
        return self.head(last_step)


D_MODEL = 64
NHEAD = 4
NUM_ENCODER_LAYERS = 2
DIM_FEEDFORWARD = 128
DROPOUT = 0.1

model = TransformerRULRegressor(
    input_size=len(feature_columns),
    d_model=D_MODEL,
    nhead=NHEAD,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    dim_feedforward=DIM_FEEDFORWARD,
    dropout=DROPOUT,
    max_len=SEQUENCE_LENGTH,
).to(device)

model


TransformerRULRegressor(
  (input_proj): Linear(in_features=24, out_features=64, bias=True)
  (pos_encoding): PositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=64, out_features=1, bias=True)
)

## 5. Training configuration (identical to E1)

Adam (lr = 1e-3), MSE loss on raw RUL cycles, batch size 256, 50 epochs, no early stopping or learning-rate scheduling — same regimen as E1, so any gap between the two notebooks' loss curves reflects the encoder, not the optimizer.


In [7]:
EPOCHS = 50
LEARNING_RATE = 1e-3

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses, val_losses = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, n_train_examples = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        n_train_examples += xb.size(0)
    train_epoch_loss = running_loss / n_train_examples

    model.eval()
    running_val_loss, n_val_examples = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            running_val_loss += loss.item() * xb.size(0)
            n_val_examples += xb.size(0)
    val_epoch_loss = running_val_loss / n_val_examples

    train_losses.append(train_epoch_loss)
    val_losses.append(val_epoch_loss)

    if epoch == 1 or epoch % 10 == 0 or epoch == EPOCHS:
        print(f"Epoch {epoch:3d}/{EPOCHS} | train MSE: {train_epoch_loss:8.3f} | val MSE: {val_epoch_loss:8.3f}")


Epoch   1/50 | train MSE: 7430.327 | val MSE: 6909.496


Epoch  10/50 | train MSE:  917.478 | val MSE:  716.740


Epoch  20/50 | train MSE:  188.389 | val MSE:  168.624


Epoch  30/50 | train MSE:  167.351 | val MSE:  155.082


Epoch  40/50 | train MSE:  152.567 | val MSE:  163.848


Epoch  50/50 | train MSE:  146.726 | val MSE:  158.078


## 6. Training/validation loss curve


In [8]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS + 1), train_losses, label="Train MSE")
plt.plot(range(1, EPOCHS + 1), val_losses, label="Validation MSE")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (RUL, cycles$^2$)")
plt.title("E2 Transformer Baseline — Training/Validation Loss (C-MAPSS FD001)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "E2_transformer_fd001_loss_curve.png", dpi=150)
plt.show()


<Figure size 800x500 with 1 Axes>

## 7. Evaluation: RMSE, MAE, and the PHM08 asymmetric score

Same scoring function as E1: for error `d = predicted - actual`, early errors (`d < 0`) cost `exp(-d/13) - 1`, late errors (`d >= 0`) cost the steeper `exp(d/10) - 1`, reflecting that underestimating RUL costs a conservative maintenance window while overestimating it risks an in-service failure. Predictions are clipped at zero before scoring.


In [9]:
def phm_score(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """PHM08 challenge asymmetric scoring function (lower is better)."""
    d = y_pred - y_true
    scores = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return float(np.sum(scores))


model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        pred = model(xb)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(yb.numpy())

y_pred = np.concatenate(all_preds).flatten()
y_true = np.concatenate(all_targets).flatten()
y_pred_clipped = np.clip(y_pred, a_min=0.0, a_max=None)

rmse = float(np.sqrt(np.mean((y_pred_clipped - y_true) ** 2)))
mae = float(np.mean(np.abs(y_pred_clipped - y_true)))
phm = phm_score(y_true, y_pred_clipped)

print(f"Test RMSE:      {rmse:.3f} cycles")
print(f"Test MAE:       {mae:.3f} cycles")
print(f"Test PHM Score: {phm:.3f}  (n={len(y_true)} engines)")


Test RMSE:      14.618 cycles
Test MAE:       10.855 cycles
Test PHM Score: 412.316  (n=100 engines)


## 8. Persisting results


In [10]:
metrics = {
    "experiment_id": "E2",
    "model": "transformer_baseline",
    "dataset": "cmapss",
    "fd_subset": "FD001",
    "seed": SEED,
    "hyperparameters": {
        "sequence_length": SEQUENCE_LENGTH,
        "d_model": D_MODEL,
        "nhead": NHEAD,
        "num_encoder_layers": NUM_ENCODER_LAYERS,
        "dim_feedforward": DIM_FEEDFORWARD,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS,
        "max_rul": 125,
    },
    "metrics": {
        "rmse": rmse,
        "mae": mae,
        "phm_score": phm,
    },
    "n_test_engines": int(len(y_true)),
}

results_path = RESULTS_DIR / "E2_transformer_fd001.json"
with open(results_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved metrics to {results_path}")
metrics


Saved metrics to /home/bruce-wayne-2005/industrial-ai-project/results/E2_transformer_fd001.json


{'experiment_id': 'E2',
 'model': 'transformer_baseline',
 'dataset': 'cmapss',
 'fd_subset': 'FD001',
 'seed': 42,
 'hyperparameters': {'sequence_length': 30,
  'd_model': 64,
  'nhead': 4,
  'num_encoder_layers': 2,
  'dim_feedforward': 128,
  'dropout': 0.1,
  'batch_size': 256,
  'learning_rate': 0.001,
  'epochs': 50,
  'max_rul': 125},
 'metrics': {'rmse': 14.6184720993042,
  'mae': 10.854952812194824,
  'phm_score': 412.3164978027344},
 'n_test_engines': 100}

## Persisting per-engine predictions

Saved alongside the aggregate metrics so downstream analysis/visualization notebooks (e.g. `EX_visualizations.ipynb`) can read per-engine true/predicted RUL directly, instead of re-running training.


In [11]:
predictions_path = RESULTS_DIR / "E2_transformer_fd001_predictions.npz"
np.savez(predictions_path, y_true=y_true, y_pred=y_pred_clipped)

print(f"Saved per-engine predictions to {predictions_path}")


Saved per-engine predictions to /home/bruce-wayne-2005/industrial-ai-project/results/E2_transformer_fd001_predictions.npz


## 9. Comparison to E1 (LSTM baseline)

Loads E1's saved metrics (if available) to directly answer this experiment's question: does self-attention improve over the recurrent baseline on FD001, under an otherwise identical pipeline?


In [12]:
e1_path = RESULTS_DIR / "E1_lstm_fd001.json"

if e1_path.exists():
    with open(e1_path) as f:
        e1_metrics = json.load(f)["metrics"]

    comparison = pd.DataFrame(
        {
            "E1 (LSTM)": e1_metrics,
            "E2 (Transformer)": metrics["metrics"],
        }
    ).T
    comparison["rmse_delta_vs_lstm"] = comparison["rmse"] - e1_metrics["rmse"]
    print(comparison)

    if metrics["metrics"]["rmse"] < e1_metrics["rmse"]:
        print("\nTransformer encoder improves test RMSE over the LSTM baseline.")
    else:
        print("\nTransformer encoder does NOT improve test RMSE over the LSTM baseline.")
else:
    print(f"E1 results not found at {e1_path} — run E1_lstm_baseline.ipynb first for a side-by-side comparison.")


                       rmse        mae     phm_score  rmse_delta_vs_lstm
E1 (LSTM)         40.532047  35.096893  18182.324219            0.000000
E2 (Transformer)  14.618472  10.854953    412.316498          -25.913575

Transformer encoder improves test RMSE over the LSTM baseline.


## Summary

Under an identical data pipeline, training regimen, and prediction head, this notebook isolates the marginal effect of replacing recurrence (E1's 2-layer LSTM) with self-attention (E2's 2-layer Transformer encoder) on FD001 RUL prediction. Results are persisted to `results/E2_transformer_fd001.json` for the eventual cross-experiment comparison table alongside E1 and the remaining planned architectures (GRU, TCN, PatchTST, Chronos fine-tune).
